# SDF Sculpting Demo

Interactive sculpting of buildings from footprints using SDF primitive recipes
(plus the optional frozen BuildingNet VQVAE neural prior).

Pipeline: `footprint + style + height + seed -> SDF recipe -> [optional VQVAE prior] -> marching cubes -> mesh`.

The footprint is preserved exactly by construction in raw procedural mode; the VQVAE prior smooths the field and trades a little footprint fidelity for organic surfaces.

Run all cells; adjust the sliders in the live-preview cell to sculpt.

In [1]:
import io, json
from pathlib import Path

import numpy as np
import torch
import trimesh
from PIL import Image
from IPython.display import display, clear_output
import ipywidgets as W

from scene.sdf_recipes import build_styled_sdf, STYLES
from scene.sdf_primitives import grid_to_mesh, polygon_bbox_with_pad, sample_grid
from scene.sdf_vqvae_prior import (
    load_buildingnet_vqvae, procedural_to_mesh_via_vqvae,
)
from scripts.render_buildingnet_objfiles import make_renderer, render_one

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

_VQVAE_CACHE = {}
def get_vqvae():
    if 'm' not in _VQVAE_CACHE:
        _VQVAE_CACHE['m'] = load_buildingnet_vqvae(device=str(DEVICE))
    return _VQVAE_CACHE['m']

_RENDERER_CACHE = {}
def get_renderer(image_size=384, scale=0.35):
    k = (image_size, scale)
    if k not in _RENDERER_CACHE:
        _RENDERER_CACHE[k] = make_renderer(DEVICE, image_size=image_size, scale=scale)
    return _RENDERER_CACHE[k]

print('imports OK; styles:', STYLES)

device: cuda
imports OK; styles: ('modern', 'colonial', 'victorian', 'industrial', 'craftsman', 'mediterranean', 'contemporary', 'public_civic')


## 1. Choose a footprint

Three quick sources:
- **Rectangle** — set width & depth.
- **L-shape** — set the outer rectangle and the notch.
- **OSM JSON** — point at a `scene/extract_osm.py` output, pick a building index.

In [2]:
def rectangle_footprint(width=10.0, depth=6.0):
    return np.array([
        [-width/2, -depth/2], [width/2, -depth/2],
        [width/2,  depth/2],  [-width/2,  depth/2],
    ], dtype=np.float32)

def l_shape_footprint(width=10.0, depth=8.0, notch_w=4.0, notch_d=4.0):
    w, d = width / 2, depth / 2
    return np.array([
        [-w, -d], [w, -d], [w, d - notch_d], [w - notch_w, d - notch_d],
        [w - notch_w, d], [-w, d],
    ], dtype=np.float32)

def osm_footprint(path, index=0):
    payload = json.load(open(path))
    poly = np.asarray(payload['buildings'][index]['polygon'], dtype=np.float32)
    # Recenter the polygon at the origin so the renderer auto-fit is consistent.
    poly = poly - poly.mean(axis=0)
    return poly

# Default footprint to start with; swap below.
current_polygon = l_shape_footprint(width=12.0, depth=8.0, notch_w=4.0, notch_d=4.0)
print('current_polygon shape:', current_polygon.shape)
print('bbox:', current_polygon.min(0), '->', current_polygon.max(0))

current_polygon shape: (6, 2)
bbox: [-6. -4.] -> [6. 4.]


## 2. Interactive sculpting

Adjust the sliders to regenerate. Each update takes ~0.2 s (procedural) / ~0.5 s (VQVAE).
Click **Save OBJ** to dump the current mesh to disk.

In [3]:
current_mesh = {'m': None}

def render_current(style, height, seed, use_vqvae, resolution):
    poly = current_polygon
    sdf_fn = build_styled_sdf(style, poly, float(height), seed=int(seed))
    if use_vqvae:
        mesh, _bb = procedural_to_mesh_via_vqvae(
            sdf_fn, poly, float(height), get_vqvae(), res=64, device=str(DEVICE),
        )
    else:
        bbox = polygon_bbox_with_pad(poly, float(height) * 2.5, pad=0.10)
        grid = sample_grid(sdf_fn, int(resolution), bbox, device=str(DEVICE))
        mesh = grid_to_mesh(grid, bbox)
    current_mesh['m'] = mesh
    if mesh is None:
        print('marching cubes returned no surface'); return
    rgb = render_one(mesh, get_renderer(image_size=384, scale=0.35), DEVICE)
    img = Image.fromarray(rgb)
    buf = io.BytesIO(); img.save(buf, format='PNG'); buf.seek(0)
    out_widget.value = buf.read()
    info_widget.value = (f'<b>style</b>={style} '
                          f'<b>h</b>={height:.1f}m <b>seed</b>={seed} '
                          f'<b>VQVAE</b>={use_vqvae} '
                          f'<b>res</b>={resolution} '
                          f'<b>V</b>={len(mesh.vertices):,} '
                          f'<b>F</b>={len(mesh.faces):,}')

def on_save(_):
    m = current_mesh['m']
    if m is None:
        print('no current mesh'); return
    out = Path(save_path.value).expanduser()
    out.parent.mkdir(parents=True, exist_ok=True)
    m.export(out)
    print(f'wrote {out}  ({len(m.faces):,} faces)')

style_w = W.Dropdown(options=list(STYLES), value='colonial', description='style')
height_w = W.FloatSlider(min=3.0, max=20.0, step=0.5, value=6.0, description='height (m)')
seed_w = W.IntSlider(min=0, max=64, step=1, value=0, description='seed')
vqvae_w = W.Checkbox(value=False, description='VQVAE prior')
res_w = W.IntSlider(min=48, max=160, step=8, value=96, description='res')
save_path = W.Text(value='/tmp/sdf_sculpt.obj', description='save to')
save_btn = W.Button(description='Save OBJ', button_style='primary')
save_btn.on_click(on_save)
info_widget = W.HTML(value='')
out_widget = W.Image(format='png', width=384, height=384)
controls = W.VBox([style_w, height_w, seed_w, vqvae_w, res_w,
                   W.HBox([save_path, save_btn]), info_widget])
ui = W.HBox([controls, out_widget])
display(ui)

def _refresh(*_):
    render_current(style_w.value, height_w.value, seed_w.value, vqvae_w.value, res_w.value)
for w in (style_w, height_w, seed_w, vqvae_w, res_w):
    w.observe(_refresh, names='value')
_refresh()

## 3. (Optional) Replace the footprint and re-sculpt

Edit one of these lines to swap the footprint, then run the interactive cell again.

In [4]:
# current_polygon = rectangle_footprint(width=14.0, depth=8.0)
# current_polygon = l_shape_footprint(width=14.0, depth=10.0, notch_w=5.0, notch_d=5.0)
# current_polygon = osm_footprint('outputs/quality_rerank_ab_sweep_lafayette4x4/east/osm_input.json', index=0)
print('polygon shape:', current_polygon.shape, 'bbox:', current_polygon.min(0), '->', current_polygon.max(0))

polygon shape: (6, 2) bbox: [-6. -4.] -> [6. 4.]


## 4. Send the current sculpt to the OSM pipeline

Once you like a sculpt, you can replay the same recipe in the full OSM-to-town pipeline:

```bash
env -u LD_PRELOAD -u LD_LIBRARY_PATH ./sdfusion/bin/python \
    scripts/osm_hunyuan_pipeline_smoke.py \
    --osm_json <your-osm.json> \
    --asset_format sdf_procedural \
    --sdf_style <pick from sliders> \
    --sdf_resolution <pick from sliders> \
    --sdf_seed_base <pick from sliders> \
    [--sdf_vqvae_prior]
```